# Hierarchiczna Analiza Skupień Metodą Warda

Hierarchiczna analiza skupień (klasteryzacja aglomeracyjna) oparta na **kryterium Warda** i odległości euklidesowej.

## Teoria i Algorytm
Metoda Warda w każdym kroku łączy parę klastrów $A$ i $B$, która minimalizuje przyrost sumy kwadratów odchyleń (SSE - Sum of Squared Errors) wewnątrz nowo powstałego klastra:
$$\Delta SSE = \frac{n_A n_B}{n_A + n_B} \|\mathbf{m}_A - \mathbf{m}_B\|^2$$
gdzie $\mathbf{m}_A, \mathbf{m}_B$ to centroidy, a $n_A, n_B$ to liczności klastrów.

Odległości po połączeniu aktualizujemy przy użyciu **rekurencji Lance'a-Williamsa**:
$$d(A \cup B, C)^2 = \alpha_A d(A,C)^2 + \alpha_B d(B,C)^2 + \beta d(A,B)^2 + \gamma |d(A,C)^2 - d(B,C)^2|$$
Dla metody Warda współczynniki te wynoszą:
$$\alpha_A = \frac{n_A + n_C}{n_A + n_B + n_C}, \quad \alpha_B = \frac{n_B + n_C}{n_A + n_B + n_C}, \quad \beta = -\frac{n_C}{n_A + n_B + n_C}, \quad \gamma = 0$$

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import scipy.cluster.hierarchy as sch

# Konfiguracja wykresów
%matplotlib inline
plt.rcParams['figure.figsize'] = (18, 5.5)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 13

# Wyłączenie domyślnego trybu interaktywnego pyplot
plt.ioff()

## 1. Główny algorytm i funkcje klastrowania
Definiujemy podstawowe funkcje odpowiedzialne za klastrowanie i wizualizację.

In [ ]:
def ward_linkage(X: np.ndarray) -> np.ndarray:
    """
    Oblicza macierz łączeń Z metodą Warda z odległością euklidesową.
    """
    X = np.asarray(X, dtype=float)
    n_samples = X.shape[0]

    active_ids = list(range(n_samples))

    # 1. Obliczamy różnice między punktami
    diff = X[:, None, :] - X[None, :, :]

    # 2. Liczymy normę (odległość) i podnosimy do kwadratu
    D = np.linalg.norm(diff, axis=-1) ** 2

    sizes = {i: 1 for i in range(n_samples)}
    id_to_pos = {i: i for i in range(n_samples)}

    next_cluster_id = n_samples
    Z = np.zeros((n_samples - 1, 4))

    for step in range(n_samples - 1):
        # 1) Najbliższa para aktywnych klastrów
        best_dist = np.inf
        best_pair = None
        for ii in range(len(active_ids)):
            for jj in range(ii + 1, len(active_ids)):
                a, b = active_ids[ii], active_ids[jj]
                d = D[id_to_pos[a], id_to_pos[b]]
                if d < best_dist:
                    best_dist = d
                    best_pair = (a, b)

        a, b = best_pair
        n_a, n_b = sizes[a], sizes[b]

        # 2) Zapis wiersza macierzy linkage (odległość = pierwiastek z kwadratu)
        lo, hi = (a, b) if a < b else (b, a)
        Z[step] = [lo, hi, np.sqrt(max(best_dist, 0.0)), n_a + n_b]

        # 3) Aktualizacja odległości wzorem Lance-Williamsa dla Warda
        new_id = next_cluster_id
        next_cluster_id += 1

        remaining = [c for c in active_ids if c != a and c != b]
        new_distances = {}
        for c in remaining:
            n_c = sizes[c]
            d_ac = D[id_to_pos[a], id_to_pos[c]]
            d_bc = D[id_to_pos[b], id_to_pos[c]]
            new_distances[c] = (
                (n_a + n_c) * d_ac + (n_b + n_c) * d_bc - n_c * best_dist
            ) / (n_a + n_b + n_c)

        # 4) Aktualizacja struktur danych
        active_ids = remaining + [new_id]
        sizes[new_id] = n_a + n_b
        del sizes[a], sizes[b]

        pos_new = id_to_pos[a]
        id_to_pos[new_id] = pos_new
        del id_to_pos[a], id_to_pos[b]

        for c in remaining:
            pos_c = id_to_pos[c]
            D[pos_new, pos_c] = new_distances[c]
            D[pos_c, pos_new] = new_distances[c]

    return Z

def fcluster_maxclust(Z: np.ndarray, n_clusters: int) -> np.ndarray:
    """
    Wyznacza przynależność do klastrów dla zadanej liczby grup.
    """
    n_merges_total = Z.shape[0]
    n_samples = n_merges_total + 1
    n_clusters = max(1, min(n_clusters, n_samples))

    parent = list(range(n_samples))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    # node_representative mapuje indeks węzła z Z (także >= n_samples,
    # czyli klastry pośrednie) na reprezentanta Union-Find
    node_representative = list(range(n_samples)) + [None] * n_merges_total

    n_merges_to_apply = n_samples - n_clusters
    for i in range(n_merges_to_apply):
        a = node_representative[int(Z[i, 0])]
        b = node_representative[int(Z[i, 1])]
        union(a, b)
        node_representative[n_samples + i] = find(a)

    roots = [find(i) for i in range(n_samples)]

    root_min_index = {}
    for i, r in enumerate(roots):
        root_min_index.setdefault(r, i)
    ordered_roots = sorted(root_min_index, key=root_min_index.get)
    label_map = {r: lbl for lbl, r in enumerate(ordered_roots, start=1)}

    return np.array([label_map[r] for r in roots], dtype=int)


In [ ]:
def leaf_order_and_positions(Z: np.ndarray):
    """Kolejność liści (DFS) i ich pozycje x na potrzeby rysowania dendrogramu."""
    n_samples = Z.shape[0] + 1
    children = {n_samples + i: (int(row[0]), int(row[1])) for i, row in enumerate(Z)}
    root = n_samples + len(Z) - 1 if len(Z) > 0 else 0

    leaf_order = []

    def dfs(node):
        if node < n_samples:
            leaf_order.append(node)
        else:
            left, right = children[node]
            dfs(left)
            dfs(right)

    dfs(root)
    leaf_x = {leaf: 10.0 * i + 5.0 for i, leaf in enumerate(leaf_order)}
    return leaf_order, leaf_x, children

def node_clusters(Z, cluster_labels, children, n_samples):
    """Określa przynależność klastrową każdego węzła drzewa.
    Dla liścia - jego etykieta klastra.
    Dla węzła wewnętrznego - etykieta jeśli wszystkie liście poddrzewa
    należą do tego samego klastra, w przeciwnym razie None.
    """
    cluster_of = {}
    for leaf in range(n_samples):
        cluster_of[leaf] = cluster_labels[leaf]
    for i, row in enumerate(Z):
        node = n_samples + i
        left, right = children[node]
        cl, cr = cluster_of.get(left), cluster_of.get(right)
        cluster_of[node] = cl if cl is not None and cl == cr else None
    return cluster_of

def plot_dendrogram(Z: np.ndarray, ax=None, color_threshold=None,
                     above_threshold_color='gray', below_threshold_color='C0',
                     labels=None, cluster_labels=None, colors=None):
    """
    Rysuje dendrogram na podstawie macierzy łączeń Z - implementacja od zera,
    bez scipy.cluster.hierarchy.dendrogram.

    Połączenia o wysokości >= color_threshold rysowane są kolorem
    `above_threshold_color`, pozostałe - `below_threshold_color`.

    Jeśli podano `cluster_labels` (wektor etykiet klastrów dla każdego liścia),
    gałęzie poniżej progu są kolorowane zgodnie z przynależnością do klastrów.
    """
    if ax is None:
        ax = plt.gca()

    n_samples = Z.shape[0] + 1
    leaf_order, leaf_x, children = leaf_order_and_positions(Z)

    # Przygotowanie kolorów klastrów
    cluster_colors = None
    cluster_of = None
    if cluster_labels is not None:
        unique = sorted(set(cluster_labels))
        if colors is not None:
            cluster_colors = {c: colors[idx % len(colors)] for idx, c in enumerate(unique)}
        else:
            palette = plt.cm.tab10(np.linspace(0, 1, len(unique)))
            cluster_colors = {c: palette[idx] for idx, c in enumerate(unique)}
        cluster_of = node_clusters(Z, cluster_labels, children, n_samples)

    node_x = dict(leaf_x)

    def get_x(node):
        if node not in node_x:
            left, right = children[node]
            node_x[node] = (get_x(left) + get_x(right)) / 2.0
        return node_x[node]

    node_y = {leaf: 0.0 for leaf in range(n_samples)}
    for i, row in enumerate(Z):
        node_y[n_samples + i] = float(row[2])

    for i, row in enumerate(Z):
        node = n_samples + i
        left, right = children[node]
        x_left, x_right = get_x(left), get_x(right)
        y_left, y_right = node_y[left], node_y[right]
        y_node = node_y[node]

        if color_threshold is not None and y_node >= color_threshold:
            cl = above_threshold_color
            ax.plot([x_left, x_left], [y_left, y_node], color=cl, linewidth=1.2)
            ax.plot([x_right, x_right], [y_right, y_node], color=cl, linewidth=1.2)
            ax.plot([x_left, x_right], [y_node, y_node], color=cl, linewidth=1.2)
        else:
            if cluster_labels is not None and cluster_of is not None:
                cl_left = cluster_of.get(left)
                cl_right = cluster_of.get(right)
                col_left = cluster_colors[cl_left] if cl_left is not None else below_threshold_color
                col_right = cluster_colors[cl_right] if cl_right is not None else below_threshold_color
                if cl_left is not None and cl_left == cl_right:
                    col_horiz = col_left
                else:
                    col_horiz = below_threshold_color
            else:
                col_left = col_right = col_horiz = below_threshold_color

            ax.plot([x_left, x_left], [y_left, y_node], color=col_left, linewidth=1.2)
            ax.plot([x_right, x_right], [y_right, y_node], color=col_right, linewidth=1.2)
            ax.plot([x_left, x_right], [y_node, y_node], color=col_horiz, linewidth=1.2)

    ax.set_xticks([leaf_x[leaf] for leaf in leaf_order])
    tick_labels = [str(leaf) for leaf in leaf_order] if labels is None else [str(labels[leaf]) for leaf in leaf_order]
    ax.set_xticklabels(tick_labels, rotation=90)
    ax.set_xlim(0, 10.0 * n_samples)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Indeks punktu")
    ax.set_ylabel("Odległość (przyrost SSE)")

    return {'leaves': leaf_order, 'leaf_positions': leaf_x}


## 2. Przygotowanie zbiorów danych
Wczytujemy i standaryzujem cztery zróżnicowane zbiory danych. Dla zbiorów o wymiarowości większej niż 2 wykonujemy rzutowanie do przestrzeni 2D przy użyciu składowych głównych (PCA), co umożliwia ich wizualizację na wykresie płaskim.

In [6]:
def load_presentation_dataset(name: str):
    """
    Wczytuje i standaryzuje wskazany zbiór danych.
    """
    scaler = StandardScaler()

    if name == 'Syntetyczny (3 chmury)':
        np.random.seed(42)
        c1 = np.random.normal(loc=[-3, -3], scale=0.8, size=(40, 2))
        c2 = np.random.normal(loc=[3, 3], scale=0.9, size=(50, 2))
        c3 = np.random.normal(loc=[-2, 4], scale=0.7, size=(45, 2))
        X = np.vstack([c1, c2, c3])
        y = np.array([0]*40 + [1]*50 + [2]*45)
        feature_names = ['Cecha X', 'Cecha Y']
        target_names = ['Chmura A', 'Chmura B', 'Chmura C']
        X_scaled = scaler.fit_transform(X)

    elif name == 'Iris (Irysy)':
        iris = load_iris()
        X, y = iris.data, iris.target
        feature_names = list(iris.feature_names)
        target_names = list(iris.target_names)
        X_scaled = scaler.fit_transform(X)

    elif name == 'Wine (Wino)':
        wine = load_wine()
        X, y = wine.data, wine.target
        feature_names = list(wine.feature_names)
        target_names = list(wine.target_names)
        X_scaled = scaler.fit_transform(X)

    elif name == 'Penguins (Pingwiny)':
        url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
        df = pd.read_csv(url)
        num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
        df = df.dropna(subset=num_cols + ['species'])

        X = df[num_cols].values
        species_cat = df['species'].astype('category')
        y = species_cat.cat.codes.values
        feature_names = num_cols
        target_names = list(species_cat.cat.categories)
        X_scaled = scaler.fit_transform(X)

    else:
        raise ValueError(f"Nieznany zbiór: {name}")

    return X_scaled, y, feature_names, target_names

## 3. Interaktywna analiza i porównanie z rzeczywistym podziałem

Panel umożliwiający dynamiczną zmianę liczby klastrów ($k$) oraz zbioru danych. Prezentowane są trzy wykresy:
1. **Dendrogram** z zaznaczonym progiem cięcia oraz kolorami gałęzi spójnymi z klastrami.
2. **Klastry (Ward)** – podział punktów przypisany automatycznie przez algorytm.
3. **Klasy rzeczywiste** – rzeczywisty podział punktów na kategorie (gatunki lub odmiany).

Przycisk **Aktualizuj wykres** służy do natychmiastowego wyczyszczenia wyjścia i ponownego wyrenderowania diagramów, co zapobiega dublowaniu wykresów w notebooku.

In [7]:
from matplotlib.figure import Figure  # Jawny import na początku komórki na wypadek braku uruchomienia komórki z importami
linkage_cache = {}

# Widgety sterujące
dataset_dropdown = widgets.Dropdown(
    options=['Syntetyczny (3 chmury)', 'Iris (Irysy)', 'Wine (Wino)', 'Penguins (Pingwiny)'],
    value='Syntetyczny (3 chmury)',
    description='Zbiór danych:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

k_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    step=1,
    description='Liczba klastrów (k):',
    continuous_update=False, # Reaguje dopiero po zwolnieniu suwaka (zapobiega lawinowym odświeżeniom)
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

run_btn = widgets.Button(
    description='Aktualizuj wykres',
    button_style='primary',
    tooltip='Kliknij, aby zresetować wyjście i odświeżyć diagramy',
    icon='refresh',
    layout=widgets.Layout(width='180px')
)

output_box = widgets.Output()

def get_linkage_matrix(dataset_name):
    if dataset_name not in linkage_cache:
        X_scaled, y, feat_names, target_names = load_presentation_dataset(dataset_name)
        Z = ward_linkage(X_scaled)
        linkage_cache[dataset_name] = (X_scaled, y, feat_names, target_names, Z)
    return linkage_cache[dataset_name]

def update_presentation(change=None):
    dataset_name = dataset_dropdown.value
    k = k_slider.value

    X_scaled, y, feat_names, target_names, Z = get_linkage_matrix(dataset_name)

    max_k = min(20, X_scaled.shape[0])
    if k_slider.max != max_k:
        k_slider.max = max_k
        if k_slider.value > max_k:
            k_slider.value = max_k
            k = max_k

    labels = fcluster_maxclust(Z, k)

    n_samples = X_scaled.shape[0]
    if k <= 1:
        color_threshold = float(Z[-1, 2] + 1e-9)
    elif k >= n_samples:
        color_threshold = 0.0
    else:
        lower = Z[n_samples - k - 1, 2]
        upper = Z[n_samples - k, 2]
        color_threshold = float((lower + upper) / 2.0)

    if X_scaled.shape[1] > 2:
        pca = PCA(n_components=2)
        X_plot = pca.fit_transform(X_scaled)
        xlabel = "Składowa Główna 1 (PCA)"
        ylabel = "Składowa Główna 2 (PCA)"
        title_suffix = " (PCA 2D)"
    else:
        X_plot = X_scaled
        xlabel = feat_names[0]
        ylabel = feat_names[1]
        title_suffix = ""

    with output_box:
        # Całkowicie czyścimy i resetujemy wyjście widgetu
        clear_output(wait=True)

        # Tworzymy figurę bez użycia pyplot (Object-Oriented API)
        # Dzięki temu backend inline Jupytera nigdy nie zduplikuje wykresu
        fig = Figure(figsize=(18, 5.5))
        ax1, ax2, ax3 = fig.subplots(1, 3)

        # Ustandaryzowana paleta kolorów Hex, aby kolory w dendrogramie, klastrach i rzeczywistych klasach były identyczne
        colors_cluster = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

        # 1. Dendrogram (ze spójnym kolorowaniem gałęzi)
        plot_dendrogram(Z, ax=ax1, color_threshold=color_threshold, cluster_labels=labels, colors=colors_cluster)
        ax1.axhline(y=color_threshold, color='red', linestyle='--', linewidth=1.5,
                    label=f'Cięcie dla k={k}')
        ax1.set_title(f"Dendrogram - {dataset_name}")
        ax1.legend(loc='upper right')
        if n_samples > 50:
            ax1.set_xticklabels([])

        # 2. Wykres klastrów
        unique_labels = np.unique(labels)
        for idx, lbl in enumerate(unique_labels):
            mask = (labels == lbl)
            ax2.scatter(
                X_plot[mask, 0],
                X_plot[mask, 1],
                color=colors_cluster[idx % len(colors_cluster)],
                s=35,
                alpha=0.85,
                edgecolors='k',
                linewidths=0.5,
                label=f'Grupa {lbl}'
            )
        ax2.set_title(f"Klastry (Ward, k={k}){title_suffix}")
        ax2.set_xlabel(xlabel)
        ax2.set_ylabel(ylabel)
        ax2.legend(loc='best')
        ax2.grid(True, linestyle=':', alpha=0.6)

        # 3. Wykres klas rzeczywistych (używa tej samej palety kolorów do ujednolicenia prezentacji)
        for idx, name in enumerate(target_names):
            mask = (y == idx)
            ax3.scatter(
                X_plot[mask, 0],
                X_plot[mask, 1],
                color=colors_cluster[idx % len(colors_cluster)],
                s=35,
                alpha=0.85,
                edgecolors='k',
                linewidths=0.5,
                label=name
            )
        ax3.set_title(f"Rzeczywiste klasy{title_suffix}")
        ax3.set_xlabel(xlabel)
        ax3.set_ylabel(ylabel)
        ax3.legend(loc='best')
        ax3.grid(True, linestyle=':', alpha=0.6)

        fig.tight_layout()
        # Wyświetlamy OO Figure bezpośrednio
        display(fig)

        # Tabela kontyngencji
        df_compare = pd.DataFrame({
            'Rzeczywiska klasa': [target_names[i] for i in y],
            'Uzyskany klaster': labels
        })
        contingency_table = pd.crosstab(df_compare['Rzeczywiska klasa'], df_compare['Uzyskany klaster'])
        print("\nTabela kontyngencji (porównanie rzeczywistego podziału z uzyskanymi klastrami):")
        display(contingency_table)

# Obsługa kliknięcia przycisku do wyzerowania/odświeżenia
def on_btn_clicked(b):
    update_presentation()

run_btn.on_click(on_btn_clicked)

# Podpięcie zdarzeń automatycznych zmian
dataset_dropdown.observe(update_presentation, names='value')
k_slider.observe(update_presentation, names='value')

# Wyświetlenie panelu (w tym przycisk Aktualizuj)
display(widgets.VBox([
    widgets.HTML("<h2>Klastrowanie Metodą Warda vs Klasy Rzeczywiste</h2>"),
    widgets.HBox([dataset_dropdown, k_slider, run_btn]),
    output_box
]))

update_presentation()